# CS 3892 / 5892 — Session 15 · Specifications in practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-15-specifications-in-practice.ipynb)

**Thursday, October 15, 2026.** One rule for one agent — *never delete without a human's
approval* — written four ways: a per-request authorization policy (**Cedar**), a policy over the
history of tool calls (**Dogwood**), temporal logic checked on a model (**NuSMV**), and
requirements written before any code (**EARS**, as Kiro uses it). Then an LLM's *answer* checked
against rules by a solver (**Automated Reasoning checks**).

**Nothing here calls AWS.** Each Python file is a small model of published semantics and says so
in its header. Every file asserts its own results. NuSMV (free, LGPL) is fetched by the setup cell.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab) and define helpers ---------------
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-15-specifications-in-practice"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

SMV = ROOT / "sessions" / SESSION / "smv"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    show(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. (Same cell as sessions 9-14.)
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older
# Colab image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import shutil

BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "03_agent_history_ltl.smv")],
                           capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path):
    """Run NuSMV on a file in batch mode; return its output minus the banner."""
    r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))

def results(out):
    """Split NuSMV output into {printed formula: (verdict, counterexample text)}.
    NuSMV does not print verdicts in file order, so match them by formula."""
    res, key = {}, None
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            key = re.sub(r"\s+", " ", m.group(1)).strip()
            res[key] = [m.group(2), ""]
        elif key and res[key][0] == "false":
            res[key][1] += line + "\n"
    return {k: tuple(v) for k, v in res.items()}

def verdicts(path):
    """Print one line per property, and return {formula: verdict}."""
    out = smv(path)
    res = results(out)
    for k, (v, _) in res.items():
        print(f"{v:6} {k}")
    return {k: v for k, (v, _) in res.items()}

print("ready -- smv(path), results(out), verdicts(path)")

## 1. One request at a time: a Cedar-style policy as a formula

A Cedar policy set decides each request on its own: **Allow** iff some `permit` matches and no
`forbid` does. No state, no loops — so the policy set is a formula over one request, and a solver
can answer questions about *every* request. (A toy model in Z3; Cedar's own analyzer uses a
verified compiler and cvc5.)

In [ ]:
out = run(PYD / "01_cedar_decision_in_z3.py")
assert "the permit is dead" in out and "v2 implies v1" in out

## 2. A policy over history: `formerly`, `previous`, `since`

Dogwood adds conditions over the events that came *before* the request: three past-time
operators, each with a window. The monitor below is what a gateway has to remember to decide
the first of them.

In [ ]:
from IPython.display import SVG, display
FIG = ROOT / "sessions" / SESSION / "figures"
display(SVG(filename=str(FIG / "monitor_formerly.svg")))

Part **A** replays the trace AWS published with Dogwood's first example and gets the same three
decisions. Part **B** is our agent: one approval, three policies, three different rows.
Part **C** is a spending limit that counts the wrong kind of event.

In [ ]:
out = run(PYD / "02_history_monitor.py")
assert out.count("DENY") >= 10 and "lets all of them through" in out

## 3. The same rule in past-time LTL, on the agent

Session 11's buggy agent, with the requirement written about the past: `Y` (one step ago) and
`O` (at some earlier step). Property (b) shows that the model's `approved` variable was a one-bit
monitor all along.

In [ ]:
display(SVG(filename=str(FIG / "agent_buggy.svg")))
f = SMV / "03_agent_history_ltl.smv"
show(f)
r = verdicts(f)
want = {"G (del -> approved)": "false",
        "G (approved <-> (del & Y asked))": "true",
        "G (del -> Y asked)": "false",
        "G (del -> O asked)": "false",
        "G ((del & O asked) -> Y asked)": "false"}
assert r == want, r

## 4. Do not fix the agent: put a monitor beside it

`S = Agent ∥ Monitor`. The agent still has its shortcut. The monitor is one bit. The gateway
carries out a delete only if the bit is set.

In [ ]:
display(SVG(filename=str(FIG / "agent_gateway.svg")))
f = SMV / "04_agent_with_gateway.smv"
show(f)
r = verdicts(f)
want = {"G (del_done -> Y asked)": "true",                       # safety: enforced
        "G (del_req -> Y asked)": "false",                       # the agent still tries
        "G (denied <-> (del_req & !( Y asked)))": "true",        # refusals are exactly the forbidden requests
        "G ( F del_done)": "false"}                              # liveness: not enforceable
assert r == want, r
print()
print("Safety holds of S although the agent is still buggy. Liveness does not:")
print(results(smv(f))["G ( F del_done)"][1])

## 5. Checking an answer against rules

The second step of Automated Reasoning checks, as four solver queries. The premises and claims
are written by hand here: no language model is called.

In [ ]:
out = run(PYD / "05_answer_check_findings.py")
for v in ("VALID", "INVALID", "SATISFIABLE", "IMPOSSIBLE"):
    assert v in out

## 6. Requirements before code: conflicts and gaps

EARS sentences as implications. The first part reproduces the example in Kiro's own post; the
second writes three requirements for our gateway and finds one conflict and one gap.

In [ ]:
out = run(PYD / "06_ears_requirements_in_z3.py")
assert "{R2, R3}" in out and "{R1, R5}" in out and "{A1, A2, A3}" in out

## Try it yourself

- In `02`, write the policy "at most two deletes in any ten minutes" with a count over a window.
  Should it count `request` events or `response` events? Build a trace that tells them apart.
- In `04`, widen the monitor's memory to two steps (`fresh` for one extra step). Which of the four
  properties change, and what run shows it?
- In `05`, add a rule that forbids deletes in production, and re-run the third question. Which
  finding do you get now?
- In `06`, add `A4: IF a delete is requested AND no approval is in force, THEN THE Gateway SHALL
  deny the delete.` Is the gap closed? Is there a new conflict?